In [ ]:
# LOAD LIBRARIES
import pandas as pd
import numpy as np
from tqdm import tqdm

pd.set_option('display.max_columns', None)

In [ ]:
# LOAD DATA
path = r"D:\CAREERS\CREDITRISK\raw"

cards = pd.read_csv(path + r"\cards_data.csv")
users = pd.read_csv(path + r"\users_data.csv")
trx   = pd.read_csv(path + r"\transactions_data.csv")

print(f"Data Loaded -> TRX: {trx.shape} | USERS: {users.shape} | CARDS: {cards.shape}")


In [ ]:
# CLEANING TRANSACTIONS DATA
trx['date'] = pd.to_datetime(trx['date'])
# Hapus simbol dolar dan koma, tangani NaN kalau ada
trx['amount'] = trx['amount'].replace('[\$,]', '', regex=True).astype(float)
trx = trx.drop_duplicates()

In [ ]:
# CLEANING USERS DATA
users.rename(columns={'id': 'client_id'}, inplace=True)
money_cols = ['per_capita_income', 'yearly_income', 'total_debt']

for col in money_cols:
    users[col] = users[col].replace('[\$,]', '', regex=True).astype(float)
users = users[users['current_age'] > 17]

In [ ]:
# CLEANING CARDS DATA
cards.rename(columns={'id': 'card_id'}, inplace=True)
# Fill NaN dengan 0 untuk debit cards yang mungkin ga punya limit
cards['credit_limit'] = cards['credit_limit'].replace('[\$,]', '', regex=True).fillna('0').astype(float)


In [ ]:
import gc 

# DELETE DUPLICATES
users = users.drop_duplicates(subset=['client_id'])
cards = cards.drop_duplicates(subset=['card_id', 'client_id'])

# DOWNCASTING 
float_cols_trx = trx.select_dtypes(include=['float64']).columns
trx[float_cols_trx] = trx[float_cols_trx].astype('float32')

float_cols_users = users.select_dtypes(include=['float64']).columns
users[float_cols_users] = users[float_cols_users].astype('float32')

float_cols_cards = cards.select_dtypes(include=['float64']).columns
cards[float_cols_cards] = cards[float_cols_cards].astype('float32')


# FIRST MERGE
df = trx.merge(users, on='client_id', how='left')
del users
gc.collect() 


# SECOND MERGE
df = df.merge(cards, on=['card_id', 'client_id'], how='left')
del cards
del trx
gc.collect()


In [ ]:
# FEATURE ENGINEERING
# Time Features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_name'] = df['date'].dt.day_name()
df['hour'] = df['date'].dt.hour

# Age Group
bins = [17, 25, 35, 45, 60, 150]
labels = ['18-25', '26-35', '36-45', '46-60', '60+']
df['age_group'] = pd.cut(df['current_age'], bins=bins, labels=labels)

# Debt Ratio
df['debt_ratio'] = np.where(
    df['yearly_income'] > 0, 
    df['total_debt'] / df['yearly_income'], 
    0
)

# Credit Utilization
df['utilization'] = np.where(
    df['credit_limit'] > 0, 
    df['amount'] / df['credit_limit'], 
    0
)

# Failed Transaction Flag
df['failed_flag'] = np.where(df['errors'].notna(), 1, 0)

# High Risk User (Skor kredit rendah & beban utang tinggi)
df['high_risk'] = np.where(
    (df['credit_score'] < 600) & (df['debt_ratio'] > 0.5), 
    1, 0
)

# Dark Web Risk
df['darkweb_risk'] = np.where(
    df['card_on_dark_web'].astype(str).str.lower().isin(['yes', 'true', '1']), 
    1, 0
)


In [ ]:
# SAVING DATA IN CHUNKS
output_path = path + r"\clean_credit_behavior.csv"
chunk_size = 50000
total_rows = len(df)

with open(output_path, 'w', encoding='utf-8-sig', newline='') as f:
    for i in tqdm(range(0, total_rows, chunk_size), desc="Saving Progress"):
        df.iloc[i:i+chunk_size].to_csv(
            f,
            index=False,
            header=(i == 0)
        )